# Homemade RecipeBowl - Image to Recipe Generator

This notebook trains an Image-to-Recipe model using ResNet50 for feature extraction and an LSTM decoder for text sequence generation.
Instructions for Kaggle:
1. Turn on GPU T4x2 (Settings -> Accelerator -> GPU T4x2).
2. Add a Food Image dataset with corresponding recipes, e.g., 'Recipe1M'.
3. Run all cells.
4. Download the `image_model.pt` at the end and place it in the `backend/models/` folder of the Flask app.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# 1. Model Architecture
class ImageToRecipeModel(nn.Module):
    def __init__(self, vocab_size, embed_size=256, hidden_size=512):
        super(ImageToRecipeModel, self).__init__()
        # CNN Encoder
        resnet = models.resnet50(pretrained=True)
        # Freeze resnet weights
        for param in resnet.parameters():
            param.requires_grad = False
            
        modules = list(resnet.children())[:-1] # remove last fc layer
        self.encoder = nn.Sequential(*modules)
        
        self.fc_encoder = nn.Linear(resnet.fc.in_features, embed_size)
        
        # RNN Decoder
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc_decoder = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, images, captions):
        features = self.encoder(images)
        features = features.view(features.size(0), -1)
        features = self.fc_encoder(features)
        
        embeddings = self.embed(captions[:, :-1]) # Ignore EOS
        # Concatenate features and embeddings 
        embeddings = torch.cat((features.unsqueeze(1), embeddings), 1)
        
        lstm_out, _ = self.lstm(embeddings)
        outputs = self.fc_decoder(lstm_out)
        return outputs

# Dummy instantiate (needs actual vocab builder)
vocab_size = 5000
model = ImageToRecipeModel(vocab_size=vocab_size).to(device)

In [ ]:
# 2. Export Dummy Model
torch.save(model.state_dict(), 'image_model.pt')
print("Model saved as image_model.pt. Expand this logic for full training with a dataset.")